# 00.07 · Free vs paid vs local: a decision framework

**Module 00 · Foundations, Tooling & Research Skills** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 60–75 min |
| 📋 Prerequisites | Lessons 00.01–00.06. |
| 🔑 Keys | None |
| 🧠 Kernel | **Python (lc-mastery)** (standard library only, no API key) |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**.
> Run cells **from top to bottom**. If anything looks strange, use **Kernel → Restart Kernel and Run All Cells…**

## Learning objectives

1. **Compare** Groq (free tier), a paid benchmark (OpenAI by default, Gemini as a low-cost option), Hugging Face Inference Providers, and local models with Ollama on cost, speed, limits, privacy, and features.
2. **Convert** "$ per 1M tokens" into "$ per notebook run" with a small cost calculator.
3. **Calculate** how many calls per minute and per day a free tier really allows, and which limit binds first.
4. **Explain** production vs preview models and free-tier data-use terms, and why each needs a ⚠️ Production note.
5. **Write** your first entry in the [model & provider decision log](../../docs/decision-log.md).

## 📚 Official reading (verified 2026-10-11)

| Page | Why |
|---|---|
| [Groq rate limits](https://console.groq.com/docs/rate-limits) · [Groq models (pricing, limits)](https://console.groq.com/docs/models) · [Your data in GroqCloud](https://console.groq.com/docs/your-data) | Free-plan limits, prices, data retention |
| [Providers and models (LangChain)](https://docs.langchain.com/oss/python/concepts/providers-and-models) | How LangChain thinks about providers |
| [HF Inference Providers](https://huggingface.co/docs/inference-providers/index) · [HF pricing and billing](https://huggingface.co/docs/inference-providers/pricing) | One token, many providers |
| [Ollama integration](https://docs.langchain.com/oss/python/integrations/providers/ollama) | Local models (Module 14) |
| [OpenAI pricing](https://developers.openai.com/api/docs/pricing) · [OpenAI: your data](https://developers.openai.com/api/docs/guides/your-data) · [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing) · [Gemini API terms](https://ai.google.dev/gemini-api/terms) | Paid benchmark and free-tier terms |
| Course: [Paid benchmark: OpenAI vs Gemini](../../docs/paid-benchmark-openai-vs-gemini.md) · [Production notes](../../docs/production-notes.md) · [Decision log](../../docs/decision-log.md) | The course's choices and why |

---
## 1. The five options at a glance

All numbers verified on **2026-10-11** on the linked pages. They change: re-check before you rely on them.

| | **Groq** (course default) | **OpenAI** (paid benchmark) | **Gemini** (low-cost / free option; Module 11 default) | **HF Inference Providers** | **Ollama** (local) |
|---|---|---|---|---|---|
| What it is | Fast hosted inference for open-weight models | Proprietary frontier models | Google's models via the Gemini Developer API | One HF token routes to many providers | Runs open models on your own machine |
| Example model | `openai/gpt-oss-20b` (production) | `gpt-6-luna` | `gemini-3.8-flash`, `gemini-3.6-flash` | many | many (Module 14) |
| Price per 1M tokens (in / out) | $0.075 / $0.30 (20B) · $0.15 / $0.60 (120B) ([models](https://console.groq.com/docs/models)) | $0.10 / $0.50 (Luna) · $2.00 / $10.00 (6.1 Sol) ([pricing](https://developers.openai.com/api/docs/pricing)) | $0.75 / $3.75 (3.8 Flash, until 2026-12-31) ([pricing](https://ai.google.dev/gemini-api/docs/pricing)) | Provider's price, billed by HF ([billing](https://huggingface.co/docs/inference-providers/pricing)) | $0 per token; you pay for hardware and electricity |
| Free tier | **Yes**: 30 RPM, 1K RPD, 8K TPM, 200K TPD for `gpt-oss-*` ([rate limits](https://console.groq.com/docs/rate-limits)) | Plan on buying credits (see the [course guide](../../docs/paid-benchmark-openai-vs-gemini.md)) | **Yes**, no billing account; limits shown per project in AI Studio, "not guaranteed" | **No monthly credits for free users**; PRO accounts get $2.00/month ([billing](https://huggingface.co/docs/inference-providers/pricing)) | Always free |
| Data use | Inference data **not retained by default**; up to 30 days of logs only for reliability/abuse investigations; Zero Data Retention available ([your data](https://console.groq.com/docs/your-data)) | API data **not used to train** OpenAI models unless you opt in; abuse-monitoring logs kept up to 30 days by default ([your data](https://developers.openai.com/api/docs/guides/your-data)) | **Free tier: prompts may be used to improve Google products and read by human reviewers**; paid tier: not used ([terms](https://ai.google.dev/gemini-api/terms)) | Depends on the underlying provider | **Never leaves your machine** |
| Strengths | Very fast, cheap, OpenAI-compatible API | Quality ceiling, mature tooling | Generous free tier, huge context | Choice, one bill | Privacy, offline, no rate limits |
| Weak spots | Tight free-tier limits; model list changes often | Costs money from the first call | Free-tier data terms; price doubles 2027-01-01 | Free tier gone; quality varies by provider | Needs RAM/GPU; smaller models are weaker |

### What "production" and "preview" mean

Groq labels models as **production** ("intended for use in your production environments") or **preview** ("intended for evaluation purposes only… may be discontinued at short notice") on its [models page](https://console.groq.com/docs/models). Google uses **stable** vs **preview** in the same way.

The course sometimes uses preview models (Groq's vision and text-to-speech models) because they are the only free option. Every such use carries a **⚠️ Production note** ([CONVENTIONS.md §9](../../CONVENTIONS.md#9-production-notes-every-shortcut-has-one)).

---
## 2. From "$ per 1M tokens" to "$ per notebook run"

In Lesson 00.05 (Part 3) you priced one call. Now let's scale that up. As a reminder, providers price **per million tokens**, with different prices for **input** (what you send) and **output** (what the model writes, including hidden *reasoning* tokens for reasoning models such as `gpt-oss`). A token is roughly ¾ of an English word.

$$\text{cost} = \frac{\text{input tokens}}{10^6} \times \text{input price} + \frac{\text{output tokens}}{10^6} \times \text{output price}$$

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Price:
    """List price in USD per 1M tokens, with where and when it was verified."""
    provider: str
    model: str
    input_per_m: float
    output_per_m: float
    status: str      # "production"/"stable" or "preview"
    source: str


# VERIFIED 2026-10-11 on the source pages. Re-check before spending money.
PRICES = [
    Price("groq", "openai/gpt-oss-20b", 0.075, 0.30, "production", "console.groq.com/docs/models"),
    Price("groq", "openai/gpt-oss-120b", 0.15, 0.60, "production", "console.groq.com/docs/models"),
    Price("groq", "qwen/qwen3.8-27b", 0.80, 4.00, "preview", "console.groq.com/docs/models"),
    Price("openai", "gpt-6-luna", 0.10, 0.50, "stable", "developers.openai.com/api/docs/pricing"),
    Price("openai", "gpt-6.1-sol", 2.00, 10.00, "stable", "developers.openai.com/api/docs/pricing"),
    Price("google", "gemini-3.8-flash", 0.75, 3.75, "stable", "ai.google.dev/gemini-api/docs/pricing (until 2026-12-31)"),
    Price("google", "gemini-3.5-flash-lite", 0.30, 2.50, "stable", "ai.google.dev/gemini-api/docs/pricing"),
]


def cost_usd(input_tokens: int, output_tokens: int, price: Price) -> float:
    """Paid-tier cost of one call (or a batch), in USD."""
    return input_tokens / 1_000_000 * price.input_per_m + output_tokens / 1_000_000 * price.output_per_m


print(f"One call with 1,000 input + 500 output tokens on gpt-oss-20b: ${cost_usd(1_000, 500, PRICES[0]):.6f}")

**You should now see** `One call with 1,000 input + 500 output tokens on gpt-oss-20b: $0.000225`. A fraction of a cent: on paid tiers, cost only matters at volume.

Now estimate a whole **notebook run**. Project 00 sends 3 prompts to 2 models. In Lesson 00.05 (Part 3) you measured one short call: a few hundred tokens, part of them reasoning. Project 00's prompts ask for longer answers, so assume about 150 input and 600 output tokens per call (reasoning models write more than you see). Replace these estimates with your own measurements whenever you have them.

In [ ]:
from IPython.display import Markdown, display

CALLS = 3                 # prompts per model in Project 00
INPUT_PER_CALL = 150      # short prompts
OUTPUT_PER_CALL = 600     # answer + reasoning tokens (an estimate; compare with what you measured in Lesson 00.05)

rows = []
for p in PRICES:
    run = cost_usd(CALLS * INPUT_PER_CALL, CALLS * OUTPUT_PER_CALL, p)
    rows.append(f"| {p.provider} | `{p.model}` | {p.status} | ${run:.5f} | {1 / run:,.0f} |")

display(Markdown(
    "| Provider | Model | Status | Paid cost per run | Runs per $1 |\n|---|---|---|---|---|\n" + "\n".join(rows)
))

**You should now see** a table with one row per model: status, paid cost per Project 00 run, and runs per $1. Every run costs well under one US cent.

### What just happened?

- On paid tiers, a whole Project 00 run costs **a fraction of a cent** on `gpt-oss-20b` or `gpt-6-luna`, and about two cents on the quality-ceiling model (`gpt-6.1-sol`). Cost is rarely the problem while learning; **rate limits** are (next section).
- **Output tokens dominate.** They cost 4–5× more than input tokens, and reasoning models generate many. Asking for shorter answers is the cheapest optimization there is.
- On Groq's and Gemini's **free tiers**, these runs cost $0; the table shows what the same work would cost once you pay.

---
## 3. What does a free tier really allow?

Rate limits come in several units ([Groq rate limits](https://console.groq.com/docs/rate-limits)): **RPM** (requests per minute), **RPD** (per day), **TPM** (tokens per minute), **TPD** (tokens per day). You are stopped by **whichever limit you hit first**, and Groq answers `429 Too Many Requests` with a `retry-after` header (the `RateLimitError` from Lesson 00.05, Part 5).

In [ ]:
@dataclass(frozen=True)
class RateLimits:
    rpm: int   # requests per minute
    rpd: int   # requests per day
    tpm: int   # tokens per minute
    tpd: int   # tokens per day


# VERIFIED 2026-10-11: Groq FREE plan, openai/gpt-oss-20b and openai/gpt-oss-120b
# (console.groq.com/docs/rate-limits, "Free Plan Limits" tab). Your org may differ: see console.groq.com/settings/limits
GROQ_FREE_GPT_OSS = RateLimits(rpm=30, rpd=1_000, tpm=8_000, tpd=200_000)


def capacity(tokens_per_call: int, limits: RateLimits) -> dict[str, object]:
    """How many calls of a given size fit per minute and per day, and which limit binds."""
    per_minute = min(limits.rpm, limits.tpm // tokens_per_call)
    per_day = min(limits.rpd, limits.tpd // tokens_per_call)
    minute_binding = "TPM (tokens)" if limits.tpm // tokens_per_call < limits.rpm else "RPM (requests)"
    day_binding = "TPD (tokens)" if limits.tpd // tokens_per_call < limits.rpd else "RPD (requests)"
    return {"calls/min": per_minute, "binds/min": minute_binding, "calls/day": per_day, "binds/day": day_binding}


for size in (200, 750, 1_500, 5_000):
    print(f"{size:>6} tokens/call → {capacity(size, GROQ_FREE_GPT_OSS)}")

**You should now see** four lines. At 200 tokens/call the **request** limits bind (30 calls/min, 1,000/day); from 750 tokens/call the **token** limits bind (10 → 1 call/min).

### What just happened?

- With tiny calls (200 tokens), the **request** limits bind: 30 per minute, 1,000 per day.
- With realistic calls (1,500 tokens: a prompt plus a reasoning answer), the **token** limits bind: about **5 calls per minute** and **133 per day**. A loop of 20 calls will hit `429` in the first minute.
- A single 5,000-token call (a long document) leaves room for only one call per minute.

This is why later lessons add **rate limiters and retries with backoff** (Lessons 01.06 and 12.02), **token budgets** (Module 04 middleware), and why Module 11 (Deep Agents, which make many long calls) defaults to Gemini's larger free tier.

---
## 4. A simple decision procedure

Real decisions weigh several factors. Start with the ones that can **rule options out** (privacy, offline), then trade off quality against cost.

In [ ]:
def choose_provider(
    *,
    sensitive_data: bool,
    must_run_offline: bool,
    needs_top_quality: bool,
    monthly_budget_usd: float,
) -> str:
    """Return a recommended starting point and the reason. A teaching heuristic, not a law.

    Args:
        sensitive_data: Personal, confidential, or regulated data is involved.
        must_run_offline: No internet, or data may not leave the machine.
        needs_top_quality: Mistakes are expensive (legal, medical, customer-facing).
        monthly_budget_usd: What you can spend per month on model calls.
    """
    # Step 1: Hard constraints first. They eliminate options.
    if must_run_offline:
        return "Ollama (local): data never leaves the machine. Validate quality with your own evals."
    if sensitive_data and monthly_budget_usd == 0:
        return ("Ollama (local), or wait for budget: free tiers with product-improvement terms "
                "(e.g. Gemini free) must not see sensitive data.")
    # Step 2: Sensitive data with a budget: the data terms decide before quality or price does.
    if sensitive_data:
        tier = "a paid frontier model" if needs_top_quality else "a paid tier"
        return (f"Use {tier} whose terms exclude training on your data, with zero data retention "
                "(ZDR) and a data processing agreement if offered (e.g. Groq with ZDR enabled). "
                "Read the provider's data page first.")
    # Step 3: Quality vs cost.
    if needs_top_quality and monthly_budget_usd > 0:
        return "Paid frontier model (e.g. OpenAI gpt-6.1-sol) with a budget alert, plus evals to prove it is needed."
    # Step 4: Default for learning and prototypes.
    return "Groq free tier (production models such as openai/gpt-oss-20b). Benchmark against a paid model when quality matters."


print(choose_provider(sensitive_data=False, must_run_offline=False, needs_top_quality=False, monthly_budget_usd=0))
print(choose_provider(sensitive_data=True, must_run_offline=False, needs_top_quality=False, monthly_budget_usd=0))
print(choose_provider(sensitive_data=False, must_run_offline=False, needs_top_quality=True, monthly_budget_usd=20))
print(choose_provider(sensitive_data=True, must_run_offline=False, needs_top_quality=True, monthly_budget_usd=20))

**You should now see** four recommendations: Groq free tier for public learning data, Ollama (local) for sensitive data with no budget, a paid model (with a budget alert) when quality is critical, and, for sensitive data that also needs top quality, a paid frontier model **under terms that exclude training, with ZDR**. The last line is the point of "hard constraints first": the data terms are checked before quality or price.

> **"Pay for quality where it matters."** The course default is free Groq models, because most learning tasks don't need more. When a lesson's stretch goal says *"compare with the paid benchmark"*, you run the same prompts on `gpt-6-luna` (or Gemini) and decide **with data** whether the difference is worth it.

---
## 5. Your first decision-log entry

Every non-trivial model or provider choice gets an entry in [`docs/decision-log.md`](../../docs/decision-log.md). The next cell drafts one for *your* setup. Edit the variables, run it, and paste the result into the file under "Add your decisions below".

In [ ]:
from datetime import date

MY_CONTEXT = "Learning Module 00 on a laptop; public data only; budget $0."
MY_DECISION = "Groq free tier, openai/gpt-oss-20b as the fast model, openai/gpt-oss-120b for harder prompts."

entry = f"""### D-101 · My default provider for Module 00 · {date.today().isoformat()}
- Context:    {MY_CONTEXT}
- Options:    Groq free tier · OpenAI gpt-6-luna (paid) · Gemini free tier · Ollama (local)
- Decision:   {MY_DECISION}
- Evidence:   Prices and limits from console.groq.com/docs/models and /docs/rate-limits (checked {date.today().isoformat()}); Lesson 00.07 capacity table.
- Trade-offs: ~5 calls/min at 1.5K tokens on the free plan; production model IDs can still be deprecated.
- Revisit:    When I need more than ~100 calls/day, private data, or Groq deprecates a model I use.
"""
print(entry)

**You should now see** a ready-to-paste `### D-101 · My default provider for Module 00 · <today>` entry with Context, Options, Decision, Evidence, Trade-offs and Revisit lines. Paste it into `docs/decision-log.md`.

---
## ⚠️ Pitfalls

| Trap | Better habit |
|---|---|
| Comparing prices without counting output and reasoning tokens | Measure real `usage_metadata` (as in Lesson 00.05) and use output-heavy estimates |
| Ignoring rate limits until a loop fails | Compute capacity first (Section 3); add retries/backoff later |
| Sending private data to a free tier | Read the data-use terms; use public or synthetic data for learning |
| Using a preview model in production | Production/GA models only, pinned IDs, tested fallback |
| Choosing the most expensive model "to be safe" | Benchmark; pay where evals show it matters |

## ⚠️ Production notes

> ⚠️ **Production note: free tiers are for learning, not for real users or real data.**
> Free tiers have no SLA, low and changeable limits, and sometimes data-use terms that allow product improvement (Gemini free tier). In production: a paid plan with a contract or SLA, a spend limit or budget alert in the provider console, Zero Data Retention where available (Groq offers it), and monitoring of 429/5xx rates. See [docs/production-notes.md](../../docs/production-notes.md).

> ⚠️ **Production note: list prices in this notebook are a snapshot.**
> The `PRICES` table is hand-copied and will go stale (Gemini 3.8 Flash doubles on 2027-01-01). In production: track actual spend from the provider's usage dashboard or billing API, and record token usage per request (Lesson 12.02).

## 🏋️ Exercises

1. **(Easy)** Add `Price("groq", "openai/gpt-oss-safeguard-20b", ...)` using the [models page](https://console.groq.com/docs/models). Is it production or preview?
2. **(Easy)** Rerun the capacity table for `tokens_per_call=3_000`. How many calls per day?
3. **(Medium)** Groq's **Developer plan** lists 250K TPM and 1K RPM for `gpt-oss-20b` on the [models page](https://console.groq.com/docs/models). Create a `RateLimits` for it (use `rpd=10**9` and `tpd=10**12` as "very large" placeholders since the page doesn't list them) and compare capacity at 1,500 tokens/call.
4. **(Stretch)** Extend `choose_provider` with a `latency_critical: bool` argument. Which option would you favor, and why?

<details>
<summary><b>Show solutions</b></summary>

**1.** `Price("groq", "openai/gpt-oss-safeguard-20b", 0.075, 0.30, "preview", "console.groq.com/docs/models")`: it is listed under **Preview Models**.

**2.** `capacity(3_000, GROQ_FREE_GPT_OSS)` → 2 calls/min, 66 calls/day (TPM and TPD bind).

**3.**
```python
GROQ_DEV_GPT_OSS_20B = RateLimits(rpm=1_000, rpd=10**9, tpm=250_000, tpd=10**12)
capacity(1_500, GROQ_DEV_GPT_OSS_20B)   # 166 calls/min: TPM still binds before RPM
```

**4.** Favor Groq (very high tokens/second) or a small local model on good hardware; measure end-to-end latency, not just tokens per second.
</details>

## ✅ Checkpoint

- [ ] You can explain why output and reasoning tokens dominate cost.
- [ ] You can compute calls per minute/day for a free tier and name the binding limit.
- [ ] You can state Gemini's free-tier data-use rule and Groq's default data retention.
- [ ] You pasted your first entry into `docs/decision-log.md`.

## 🔎 Research checkpoint

Open [Groq rate limits](https://console.groq.com/docs/rate-limits) and then your own [limits page](https://console.groq.com/settings/limits) (requires login). Are your organization's limits the same as the published free-plan table? Log the answer in `docs/research-log.md`.

### 🔎 Module 00 research checkpoint

This is the last lesson of the module, so close it with the module-level entry. Open [Groq deprecations](https://console.groq.com/docs/deprecations) and the [LangChain changelog](https://docs.langchain.com/oss/python/releases/changelog). Log **one model deprecated in the last 90 days** and **one new LangChain feature** in [`docs/research-log.md`](../../docs/research-log.md), using the template at the top of that file.

## Summary

- Groq (free, fast) is the course default; OpenAI `gpt-6-luna` is the paid benchmark; Gemini is the low-cost option (and Module 11's default); HF routes to many providers; Ollama keeps data local.
- Cost = tokens × price, and output tokens dominate. Free tiers are limited mostly by **tokens per minute**.
- Privacy and offline needs eliminate options before price does.
- Record decisions with evidence, so you know when to revisit them.

**Next:** [Project 00 · Groq Model Explorer](project/00-project-groq-model-explorer.ipynb), then [Module 01](../01-models-messages-prompts/).

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 00 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)